# 03 - Model Explainability

Notebook doc va hien thi:
- Metrics cua 3 mo hinh
- SHAP values cho XGBoost
- Feature Importance
- ROC Curve, Precision-Recall Curve
- Confusion Matrix
- Threshold Tuning

## 1. Import thu vien va load data

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score
)

sns.set_theme(style="whitegrid")

## 2. Load metrics va artifacts

In [ ]:
# Load metrics
with open(ROOT / 'reports' / 'metrics.json', encoding='utf-8') as f:
    metrics = json.load(f)

# Load feature importance
with open(ROOT / 'reports' / 'feature_importance.json', encoding='utf-8') as f:
    feature_importance = json.load(f)

print("Metrics:")
metrics

In [ ]:
print("\nTop 10 Feature Importance:")
print("\nRandom Forest:")
for i, item in enumerate(feature_importance['random_forest'][:10], 1):
    print(f"  {i}. {item['feature']}: {item['importance']:.4f}")

print("\nXGBoost:")
for i, item in enumerate(feature_importance['xgboost'][:10], 1):
    print(f"  {i}. {item['feature']}: {item['importance']:.4f}")

## 3. Load model va test data

In [ ]:
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()

models = {
    "logistic_regression": joblib.load(MODELS_DIR / "logistic_regression.joblib"),
    "random_forest": joblib.load(MODELS_DIR / "random_forest.joblib"),
    "xgboost": joblib.load(MODELS_DIR / "xgboost.joblib")
}

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

## 4. Bang so sanh metrics

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": name.replace("_", " ").title(),
        "Accuracy": data["accuracy"],
        "Precision": data["precision"],
        "Recall": data["recall"],
        "F1": data["f1"],
        "ROC-AUC": data["roc_auc"]
    }
    for name, data in metrics.items()
])

comparison_df

In [ ]:
def highlight_best(s):
    return ['font-weight: bold' if v == s.max() else '' for v in s]

styled_df = comparison_df.set_index("Model").style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
    "ROC-AUC": "{:.4f}"
}).background_gradient(subset=["ROC-AUC", "F1", "Recall"], cmap="Blues")

styled_df

## 5. ROC Curve Comparison

In [ ]:
colors = {"logistic_regression": "#7c3aed", "random_forest": "#2563eb", "xgboost": "#16a34a"}

plt.figure(figsize=(8, 6))

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, 
             label=f"{name.replace('_', ' ').title()} AUC = {roc_auc:.4f}",
             color=colors[name], linewidth=2)

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Classifier")
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curve Comparison", fontsize=14)
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "roc_curve_comparison.png", dpi=150)
plt.show()

## 6. Precision-Recall Curve

In [ ]:
plt.figure(figsize=(8, 6))

for name, model in models.items():
    y_proba = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    plt.plot(recall, precision, 
             label=f"{name.replace('_', ' ').title()} AP = {ap:.4f}",
             color=colors[name], linewidth=2)

plt.xlabel("Recall", fontsize=12)
plt.ylabel("Precision", fontsize=12)
plt.title("Precision-Recall Curve Comparison", fontsize=14)
plt.legend()
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "precision_recall_curve.png", dpi=150)
plt.show()

In [ ]:
print("\nNhan xet Precision-Recall:")
print("- Neu muc tieu la sang loc nguy co, recall cao thuong quan trong hon.")
print("- Precision qua thap se tao nhieu canh bao nham.")
print("- Can can bang giua recall va precision tuy theo muc tieu nghiep vu.")

## 7. Feature Importance Comparison

In [ ]:
def get_transformed_feature_names(pipeline):
    preprocessor = pipeline.named_steps["preprocessor"]
    names = preprocessor.get_feature_names_out()
    return [
        name.replace("continuous_scaler__", "")
           .replace("category_encoder__", "")
           .replace("remainder__", "")
        for name in names
    ]

rf_features = get_transformed_feature_names(models["random_forest"])
rf_importance = models["random_forest"].named_steps["classifier"].feature_importances_

xgb_features = get_transformed_feature_names(models["xgboost"])
xgb_importance = models["xgboost"].named_steps["classifier"].feature_importances_

rf_df = pd.DataFrame({"feature": rf_features, "importance": rf_importance, "model": "Random Forest"})
xgb_df = pd.DataFrame({"feature": xgb_features, "importance": xgb_importance, "model": "XGBoost"})

importance_compare = pd.concat([rf_df, xgb_df])

top_features = (
    importance_compare.groupby("feature")["importance"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
    .index
)

In [ ]:
plt.figure(figsize=(11, 7))
sns.barplot(
    data=importance_compare[importance_compare["feature"].isin(top_features)],
    x="importance",
    y="feature",
    hue="model",
    palette={"Random Forest": "#2563eb", "XGBoost": "#16a34a"}
)
plt.title("Feature Importance Comparison: Random Forest vs XGBoost", fontsize=14)
plt.xlabel("Importance", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.legend(title="Model")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "feature_importance_comparison.png", dpi=150)
plt.show()

In [ ]:
print("\nNhan xet Feature Importance:")
print("- HighBP, GenHlth, BMI thuong la cac feature quan trong nhat.")
print("- Random Forest va XGBoost co su dong thuan ve feature quan trong.")
print("- Backend co the cho phep xem feature importance theo tung mo hinh.")

## 8. Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cmaps = {"logistic_regression": "Purples", "random_forest": "Blues", "xgboost": "Greens"}

for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, ax=ax, cmap=cmaps[name]
    )
    ax.set_title(name.replace("_", " ").title(), fontsize=13)

plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
def confusion_items(cm):
    tn, fp = cm[0]
    fn, tp = cm[1]
    return {"TN": tn, "FP": fp, "FN": fn, "TP": tp}

confusion_bar_df = pd.DataFrame([
    {"model": name.replace("_", " ").title(), **confusion_items(m["confusion_matrix"])}
    for name, m in metrics.items()
])

confusion_long = confusion_bar_df.melt(
    id_vars="model",
    value_vars=["TP", "FN", "FP", "TN"],
    var_name="Type",
    value_name="Count"
)

plt.figure(figsize=(11, 6))
sns.barplot(data=confusion_long, x="Type", y="Count", hue="model", palette="Set2")
plt.title("Confusion Matrix Bar Comparison", fontsize=14)
plt.xlabel("Confusion Type", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.legend(title="Model")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "confusion_matrix_bar.png", dpi=150)
plt.show()

In [ ]:
print("\nChi tiet Confusion Matrix:")
for name, m in metrics.items():
    tn, fp = m["confusion_matrix"][0]
    fn, tp = m["confusion_matrix"][1]
    print(f"\n{name.replace('_', ' ').title()}:")
    print(f"  TN (True Negative): {tn:,}")
    print(f"  FP (False Positive): {fp:,} - Du doan nham diabetic")
    print(f"  FN (False Negative): {fn:,} - Bo sot ca diabetic")
    print(f"  TP (True Positive): {tp:,}")

## 9. SHAP Explainability (XGBoost)

In [ ]:
import shap

xgb_model = models["xgboost"]
xgb_classifier = xgb_model.named_steps["classifier"]

X_test_transformed = xgb_model.named_steps["preprocessor"].transform(X_test)
feature_names = get_transformed_feature_names(xgb_model)

print(f"Feature names: {len(feature_names)}")
print(f"Transformed X_test shape: {X_test_transformed.shape}")

In [ ]:
print("Dang tao SHAP explainer...")
explainer = shap.TreeExplainer(xgb_classifier)

sample_size = min(5000, X_test_transformed.shape[0])
X_sample = X_test_transformed[:sample_size]

print(f"Dang tinh SHAP values cho {sample_size} mau...")
shap_values = explainer.shap_values(X_sample)

print("Da xong!")

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
plt.title("SHAP Summary Plot - XGBoost", fontsize=14)
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "shap_summary.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, plot_type="bar", show=False)
plt.title("SHAP Feature Importance - XGBoost", fontsize=14)
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "shap_bar.png", dpi=150, bbox_inches='tight')
plt.show()

## 10. Threshold Tuning

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, confusion_matrix

y_proba_xgb = models["xgboost"].predict_proba(X_test)[:, 1]

print("So sanh cac threshold khac nhau cho XGBoost:\n")

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

for thresh in thresholds:
    y_pred = (y_proba_xgb >= thresh).astype(int)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp = cm[0]
    fn, tp = cm[1]
    
    print(f"Threshold: {thresh}")
    print(f"  Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    print(f"  TP: {tp:,}, FN: {fn:,}, FP: {fp:,}, TN: {tn:,}")
    print()

In [ ]:
print("\nNhan xet Threshold Tuning:")
print("- Threshold thap hon (0.3-0.4): Recall cao hon, phat hien nhieu ca diabetic hon nhung FP tang.")
print("- Threshold cao hon (0.6-0.7): Precision cao hon, it canh bao nham nhung co the bo sot nhieu ca.")
print("- Muc dich sang loc: nen uu tien recall cao.")
print("- Muc dich xac dinh chinh xac: nen uu tien precision cao.")
print("- Default threshold 0.5 la diem can bang.")

## 11. Ket luan

In [ ]:
print("""
KET LUAN MODEL EXPLAINABILITY:
================================
1. ROC-AUC cua XGBoost thuong cao nhat trong 3 mo hinh.
2. Feature quan trong nhat: HighBP, GenHlth, BMI, DiffWalk, HighChol.
3. SHAP giup giai thich tung prediction cu the.
4. Threshold co the dieu chinh tuy theo muc tieu nghiep vu.

CAC HINH DA LUU:
- roc_curve_comparison.png
- precision_recall_curve.png
- feature_importance_comparison.png
- confusion_matrix.png
- confusion_matrix_bar.png
- shap_summary.png
- shap_bar.png

HUONG DAN SU DUNG:
1. Chay Backend: cd backend && python app.py (port 5000)
2. Mo Frontend: python -m http.server 8000 (port 8000)
3. Truy cap: http://localhost:8000/frontend/
""")